# 🧬 Vectors, 3D rotations, and coordinate systems

**Xu Lab | Biology-to-CryoEM Research | Module 03**  
**Phase:** A: Biology + computation | **Level:** scaffolded undergraduate → research | **Time:** 60–100 minutes  
**Hardware:** Google Colab free-tier CPU (optional T4 GPU for later extensions)  
**Core tools:** Python, NumPy, SciPy, Matplotlib; selected modules use scikit-learn/PyTorch  
**Prerequisites:** Previous tutorial  
**Status:** self-contained *pedagogical demonstration* using synthetic data, not an official reproduction of Xu Lab publications.  
**Safety:** for research and educational purposes only, not for clinical diagnosis.

## Learning objectives

1. Explain the biological question and physical limitations.
2. Derive or interpret the model below and identify its assumptions.
3. Run the experiment, change a parameter, and interpret quantitative output.
4. Explain how the concept supports the research papers linked below.

## Biological motivation

The same particle appears under different orientations; aligning those appearances is central to averaging.

## Conceptual bridge

Build rigid rotations, verify that rotations preserve distances, and distinguish translation from orientation.

## Mathematical model

$\mathbf x\prime=R\mathbf x+\mathbf t$, with $R^TR=I$ and $\det R=1$.

## Notebook roadmap

1. Reproducible setup  2. Synthetic experiment  3. Interpretation  4. Challenge  5. Paper connection.


In [ ]:
# -- Environment check and shared utilities --
import sys, importlib.util
import numpy as np
import scipy
from scipy import ndimage as ndi, signal
import matplotlib.pyplot as plt
SEED=2026
rng=np.random.default_rng(SEED)
plt.rcParams.update({'figure.dpi':95,'axes.grid':False})
print('Python:',sys.version.split()[0], 'NumPy:',np.__version__)
print('PyTorch available:',bool(importlib.util.find_spec('torch')))
try:
    import torch
    print('CUDA available:',torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch not installed; required only for Module 13/14')
# Colab stock environment normally contains the required libraries.
# If an import fails, run: %pip install -q numpy scipy matplotlib scikit-learn torch

def phantom(n=48):
    """Build a deterministic 3D synthetic density with three distinct macromolecular blobs."""
    zz, yy, xx = np.mgrid[:n,:n,:n].astype(float)
    rho = np.zeros((n,n,n),dtype=float)
    for z,y,x,s,amp in [(0.45*n,.47*n,.48*n,.10*n,1.),(.65*n,.38*n,.58*n,.07*n,.7),(.40*n,.66*n,.36*n,.075*n,.8)]:
        rho += amp*np.exp(-((zz-z)**2+(yy-y)**2+(xx-x)**2)/(2*s*s))
    return rho/np.max(rho)


## Experiment: build, inspect, and check a scientific claim

Run the code below, then identify (i) input, (ii) transformation, (iii) output and (iv) a key assumption.

In [ ]:
from scipy.spatial.transform import Rotation
xyz=np.array([[1.,0.,0.],[0.,1.,0.],[0.,0.,1.]])
rot=Rotation.from_euler('z',40,degrees=True).as_matrix()
y=xyz@rot.T
assert np.allclose(np.linalg.norm(y,axis=1),np.linalg.norm(xyz,axis=1))
assert np.allclose(rot@rot.T,np.eye(3))
print('Rotation matrix:',np.round(rot,2))

## Explain what happened

**Expected scientific observation:** Rotations change orientation but preserve pairwise distances; translation-only correction leaves rotational mismatch.

Write 3–5 sentences connecting the plotted result to the biological motivation. Describe one failure case and one alternative explanation.

## Guided challenge (edit code above)

Create a 90° rotation and determine which axes are permuted. Why is it invalid to align particles by translations alone?

**Deliverable:** a labeled figure or small results table, a claim, and one limitation. Quantify the effect rather than just saying “it looks better.”

## Research connection and reading strategy

**Related Xu Lab research:** 2009 rotation-invariant features; 2020 Gum-Net; 2025 BOE-ViT.

Read each paper in this order: biological question → observations/data → forward model → key objective → baseline → evaluation split → limitations. Distinguish what the toy exercise illustrates from the actual published implementation. See `../PAPER_TO_TUTORIAL_MAP.md` for publication titles and further prerequisites.

## Exit ticket

1. What is known from the measured data, versus assumed by the model?
2. Which plotted quantity could be misleading if evaluated only on synthetic data?
3. Sketch one controlled ablation and one biological validation experiment.
4. Save an updated notebook and a 150-word lab note.